# 🚨 Fraud Detection Agent — LangChain ReAct Edition

This notebook demonstrates the **complete fraud detection system** powered by a **real LangChain ReAct agent**.

## Features Included ✨

### 1. **Real Agentic Loop** 🤖
   - LangGraph `create_react_agent` with Mistral AI via OpenRouter
   - The LLM autonomously decides which tools to call and in what order
   - True reasoning — not a hardcoded pipeline

### 2. **Account & Card Analysis** 📊
   - Transaction scoring using Autoencoder + LSTM models
   - Risk calculation with weighted metrics
   - Anomaly detection ratios

### 3. **Graph RAG Insights** 🧩
   - Neo4j network-level signals
   - Shared accounts, suspicious connections, community patterns

### 4. **LLM-Generated Explanation** 💡
   - Full narrative generated by the agent after tool calls
   - Human-readable fraud reasoning

## How to Use

1. Run the single code cell below to launch the Gradio interface
2. Enter queries like: `"Check fraud for client181"`
3. Get a concise BANK INVESTIGATOR REPORT

### Example queries:
- `"Check fraud for client181"`
- `"Analyze client962 suspicious transactions"`
- `"Verify client181 card fraud on 2021-01-04"`
- `"Check fraud for client100 in 2021"`

---

In [1]:
# Single consolidated Gradio cell: imports, agent hot-reload, analyze function, interface, launch
import gradio as gr
import importlib
from datetime import datetime
import socket
import logging
import services.logger
logger = logging.getLogger(__name__)

# Hot-reload agent modules so edits are picked up without restarting the kernel
import agent.langchain_fraud_agent
import agent.simple_text_agent
importlib.reload(agent.langchain_fraud_agent)
importlib.reload(agent.simple_text_agent)
from agent.simple_text_agent import fraud_agent_text

# MAIN ANALYSIS FUNCTION (BANK REPORT + GRAPH VISUALIZATION)
def analyze_fraud_comprehensive(query: str):
    """
    Runs the LangChain ReAct fraud agent and returns two outputs:
      - BANK INVESTIGATOR REPORT (Markdown)
      - Graph Visualization (HTML iframe)
    """
    if not query or not query.strip():
        return ('⚠️ Please enter a query, e.g., **Check fraud for client181**', '')

    try:
        # Run the agent
        result = fraud_agent_text(query)

        # If the agent returned an error, show it in the report and no viz
        if isinstance(result, dict) and result.get('error'):
            return (f'❌ **Agent Error**\n\n**Error:** {result.get("error")}', '')

        # Extract judge report (added by the agent)
        bank_report = result.get('bank_report') if isinstance(result, dict) else None

        # Extract visualization HTML (if present)
        viz_html = ''
        viz_iframe = ''
        if isinstance(result, dict):
            graph = result.get('graph', {})
            if isinstance(graph, dict):
                viz_html = graph.get('viz_html', '') or ''

        # Wrap visualization HTML in a data URL iframe so Gradio will render it
        if viz_html:
            try:
                import base64
                b64 = base64.b64encode(viz_html.encode('utf-8')).decode('ascii')
                viz_iframe = f'<iframe src="data:text/html;base64,{b64}" width="100%" height="480" style="border:0"></iframe>'
            except Exception:
                viz_iframe = viz_html

        # Format only the bank investigator report
        if isinstance(bank_report, dict):
            if bank_report.get('final_decision'):
                fd = bank_report.get('final_decision')
                fc = bank_report.get('final_confidence', 0)
                action = bank_report.get('recommended_action', '')
                summary_j = bank_report.get('summary', '')
                evidence = bank_report.get('evidence', [])

                md = f'# 🧾 BANK INVESTIGATOR REPORT\n\n'
                md += f'**Final Decision:** **{fd}**  \\n'
                md += f'**Confidence:** {fc:.0%}  \\n\n'
                md += f'**Recommended Action:** {action}  \\n\n'
                md += f'**Summary:**  \\n{summary_j}\\n\\n'

                if evidence:
                    md += '**Evidence:**\\n'
                    for ev in evidence:
                        md += f'- {ev}\\n'
            elif bank_report.get('raw_llm'):
                md = f'**Judge Output (raw):**\\n\\n{bank_report.get("raw_llm")}'
            elif bank_report.get('error'):
                md = f'**Judge Error:** {bank_report.get("error")}'
            else:
                md = '**No judge report available.**'
        elif bank_report:
            md = str(bank_report)
        else:
            md = '**No judge report available.**'

        md += f'\\n\\n---\\n_Report generated: {datetime.now().isoformat()}_'
        return (md, viz_iframe)

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return (f'❌ **Unexpected Error**\\n\\n**Type:** `{type(e).__name__}`\\n**Message:** {str(e)}\\n\\n```\\n{tb}\\n```', '')

# Helper: find a free local port to avoid collisions
def _find_free_port(start=7860, end=7899):
    s = socket.socket()
    for port in range(start, end):
        try:
            s.bind(('127.0.0.1', port))
            s.close()
            return port
        except OSError:
            continue
    return 7860

port = _find_free_port()

# Build and launch the Gradio interface (single cell)
demo = gr.Interface(
    fn=analyze_fraud_comprehensive,
    inputs=gr.Textbox(lines=2, placeholder='Enter query, e.g., Check fraud for client181'),
    outputs=[gr.Markdown(label='BANK INVESTIGATOR REPORT'), gr.HTML(label='Graph Visualization')],
    title='Fraud Detection Agent - BANK INVESTIGATOR REPORT',
    description='Runs the agent and returns the bank investigator report and graph visualization.',
)

try:
    # Try inline display (classic Jupyter). If unsupported, fallback below.
    demo.launch(share=False, inline=True, server_port=port)
except Exception as e:
    logger.exception('Inline Gradio display not available; launching external server: %s', e)
    demo.launch(share=False, inbrowser=True, prevent_thread_lock=True, server_port=port)


2026-04-18 22:06:18,003 [INFO] numexpr.utils: NumExpr defaulting to 12 threads.
2026-04-18 22:06:20,049 [INFO] db.mongo: Connected to MongoDB (db=fraud_db)
2026-04-18 22:06:20,688 [INFO] model.card_model: Card models loaded from c:\Users\moham\Music\fraud_agent_final_14_04_2026_v3\model\..\models_card
2026-04-18 22:06:20,696 [INFO] model.card_model: Card models loaded from c:\Users\moham\Music\fraud_agent_final_14_04_2026_v3\model\..\models_card
2026-04-18 22:06:20,697 [INFO] model.account_model: Loading models from saved_models
2026-04-18 22:06:20,702 [INFO] model.account_model: Models loaded and cached from saved_models
* Running on local URL:  http://127.0.0.1:7860
2026-04-18 22:06:22,160 [INFO] httpx: HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-04-18 22:06:22,180 [INFO] httpx: HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
* To create a public link, set `share=True` in `launch()`.


2026-04-18 22:06:24,294 [INFO] httpx: HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-04-18 22:07:49,866 [INFO] agent.langchain_fraud_agent: 🤖 LangChain ReAct Agent starting for client962
2026-04-18 22:07:49,868 [INFO] agent.langchain_fraud_agent: Query: Analyze client962 transactions
2026-04-18 22:07:51,171 [INFO] httpx: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-18 22:07:53,444 [INFO] httpx: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-18 22:07:53,938 [ERROR] graph.neo4j_connection: Neo4j connectivity test failed: Failed to DNS resolve address 70ee550a.databases.neo4j.io:7687: [Errno 11001] getaddrinfo failed
Traceback (most recent call last):
  File "c:\Users\moham\anaconda3\Lib\site-packages\neo4j\_async_compat\network\_util.py", line 180, in _dns_resolver
    info = NetworkUtil.get_address_info(
        address.host,
    ...<2 lines>...
        type=socket.SOCK